# RAG Law Chatbot — Improvement Notebook

Notebook này cải tiến hệ thống RAG baseline để trả lời câu hỏi pháp luật tiếng Việt tốt hơn.

**Các cải tiến chính so với baseline:**

1. **Chunking**: Giảm `chunk_size` từ 2000 xuống 1024 ký tự để tăng độ chính xác khi retrieval
2. **Embedding**: Thêm `normalize_embeddings=True` và đổi distance strategy sang Cosine Similarity thay vì L2 mặc định
3. **Retrieval**: Dùng Hybrid Search (FAISS dense + BM25 lexical) thay vì chỉ dùng FAISS
4. **Reranking**: Thêm CrossEncoder reranker (`BAAI/bge-reranker-large`) để sắp xếp lại kết quả retrieval
5. **LLM**: Nâng cấp từ `Qwen/Qwen1.5-1.8B` lên `Qwen/Qwen3-1.7B`

---




## Phần 1: Cài đặt thư viện

In [19]:
!pip install -q langchain-community langchain-google-genai langchain-core faiss-cpu pandas tqdm
!pip install --upgrade langchain
!pip install --upgrade langchain-text-splitters
!pip install --upgrade tqdm
!pip install --upgrade langchain_huggingface
!pip install sentence-transformers
!pip install scikit-learn

In [20]:
!pip install faiss-cpu
from langchain_community.vectorstores import FAISS

## Phần 2: Kết nối Google Drive


In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
%cd /content/drive/MyDrive/LawChatbot/

/content/drive/MyDrive/LawChatbot


## Phần 3: Tải dữ liệu

Giải nén bộ dữ liệu pháp luật (~16,440 file `.txt`) từ `Dataset.zip`.


In [23]:
!unzip -n "/content/drive/MyDrive/LawChatbot/Dataset.zip" -d "/content/drive/MyDrive/LawChatbot/export"

Archive:  /content/drive/MyDrive/LawChatbot/Dataset.zip


### 3.1 Tải file RES dataset (câu hỏi và đáp án)


In [26]:
from google.colab import drive
import pandas as pd
import os

res_path = '/content/drive/MyDrive/LawChatbot/RES.csv'
SHEET_ID = "1nD9gzxp5LalIg4BQwpZVKOoAVj8YUpSs"
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv"

df = pd.read_csv(csv_url)
# csv_output = "/content/drive/MyDrive/LawChatbot/RES.csv"
df.to_csv(res_path, index=False)



In [27]:
df = pd.read_csv(res_path)
df.head()

,Câu Hỏi,Trả lời,Số hiệu VBPL (Trích xuất),Câu trả lời của URAx-Law,Reference có đúng không?,Trả lời có đúng không?,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,"Soft OTP là hình thức xác thực thế nào, gồm nh...",Căn cứ tại điểm d khoản 3 Điều 11 Thông tư 50/...,50/2024/TT-NHNN,### Soft OTP là gì?\n \n Soft OTP (One Time Pa...,1.0,1.0,NaN,NaN,NaN,NaN,Hạng mục,Số lượng
1,Những nội dung tối thiểu gì mà tổ chức cung cấ...,Căn cứ tại khoản 2 Điều 17 Thông tư 50/2024/TT...,50/2024/TT-NHNN,"Theo quy định, tổ chức cung cấp dịch vụ Online...",1.0,1.0,NaN,NaN,NaN,NaN,Tổng số câu hỏi,2748
2,"Theo quy định hiện hành, đơn vị cung cấp dịch ...",Căn cứ theo Điều 15 Thông tư 50/2024/TT-NHNN q...,50/2024/TT-NHNN,### Hệ thống giám sát và theo dõi hoạt động tr...,1.0,1.0,NaN,NaN,NaN,NaN,Số câu trả lời được,132
3,"Khái niệm ""hệ quản trị cơ sở dữ liệu"" được Thô...",Căn cứ tại khoản 10 Điều 2 Thông tư 50/2024/TT...,50/2024/TT-NHNN,"**Khái niệm ""hệ quản trị cơ sở dữ liệu""**\n \n...",1.0,1.0,NaN,NaN,NaN,NaN,Tổng điểm của số câu trả lời đúng reference,122
4,"Từ ngày 01/01/2025, các yêu cầu bắt buộc đối v...",Căn cứ theo Điều 6 Thông tư 50/2024/TT-NHNN qu...,50/2024/TT-NHNN,"Từ ngày 01/01/2025, theo Thông tư số 50/2024/T...",1.0,1.0,NaN,NaN,NaN,NaN,Tổng điểm câu trả lời đúng,132


---

## Phần 4: Ingestion — Xây dựng Vector Store

**Thay đổi so với baseline:**

- `chunk_size`: 2000 giảm xuống 1024 — chunks nhỏ hơn giúp retrieval chính xác hơn
- Distance strategy: L2 đổi sang Cosine Similarity — phù hợp hơn cho semantic search
- `normalize_embeddings=True` được thêm vào để vector được chuẩn hóa trước khi index vào FAISS
- Vector store đã được build sẵn và lưu tại Google Drive để tiết kiệm thời gian chạy


**Chọn cách load vector store**

Có 2 lựa chọn:

- **Tải sẵn** (khuyến nghị): Tải vector store `vietnamese-sbert_1024_cosine` đã build trước
- **Build lại từ đầu**: Chạy lại toàn bộ ingestion pipeline (tốn khoảng 30–60 phút)

### 4.1 Tải vector store có sẵn


* Tải các vector store vietnamese-sbert_1024_cosine đã build sẵn

* vietnamese-sbert_1024_cosine là vector store sửa dụng độ đo cosine thay vì L2 mặc định trong baseline, độ dài mỗi chunk là 1024

In [28]:
%cd /content/drive/MyDrive/LawChatbot/

/content/drive/MyDrive/LawChatbot


In [29]:
!pip install -q gdown
!gdown "https://drive.google.com/uc?id=1U8Zq29iwrbc6sk8uGtFSPHc2TnoaGjan"


Downloading...
From (original): https://drive.google.com/uc?id=1U8Zq29iwrbc6sk8uGtFSPHc2TnoaGjan
From (redirected): https://drive.google.com/uc?id=1U8Zq29iwrbc6sk8uGtFSPHc2TnoaGjan&confirm=t&uuid=ccf51b58-8777-4d96-8a14-74e523e8fc9e
To: /content/drive/MyDrive/LawChatbot/vietnamese-sbert_1024_cosine.zip
100% 1.07G/1.07G [00:23<00:00, 45.5MB/s]


In [30]:
import os

folder_path = "/content/drive/MyDrive/LawChatbot/vietnamese-sbert_1024_cosine"

if not os.path.exists(folder_path):
    !unzip vietnamese-sbert_1024_cosine.zip -d /content/drive/MyDrive/LawChatbot/
    print("Đã giải nén xong.")
else:
    print("Folder đã tồn tại, bỏ qua bước giải nén.")

Folder đã tồn tại, bỏ qua bước giải nén.


### 4.2 (Tùy chọn) Build vector store từ đầu

Bỏ comment toàn bộ cell bên dưới nếu muốn build lại từ đầu phần vector store.


In [ ]:
# import os
# from tqdm import tqdm
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import TextLoader
# from langchain_community.vectorstores import FAISS
# from langchain_community.embeddings import HuggingFaceEmbeddings

# # === Configuration ===
# DOCUMENTS_PATH = "/content/drive/MyDrive/LawChatbot/export/export_1"
# EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"

# BATCH_SIZE = 128
# OUTPUT_DIR = "/content/drive/MyDrive/LawChatbot/vietnamese-sbert_1024_cosine/"

# # === Load text documents ===
# def load_text_documents(path):
#   documents = []
#   for filename in os.listdir(path):
#       if filename.endswith(".txt"):
#           file_path = os.path.join(path, filename)
#           loader = TextLoader(file_path, encoding='utf-8')
#           raw_docs = loader.load()
#           for doc in raw_docs:
#               # Normalize content
#               content = doc.page_content.replace("\n", " ").lower().strip()
#               doc.page_content = content
#               documents.append(doc)
#   return documents

# print("Loading text documents...")
# raw_documents = load_text_documents(DOCUMENTS_PATH)
# print(f"Total documents found: {len(raw_documents)}")

# subset_idx = len(raw_documents)
# raw_documents = raw_documents[:subset_idx]
# print(f"Documents selected for embedding: {len(raw_documents)}")

# # === Split documents into smaller chunks ===
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1024,
#     chunk_overlap=50,
#     length_function=len
# )

# texts = []
# for doc in tqdm(raw_documents, desc="Splitting documents into chunks"):
#     chunks = text_splitter.split_documents([doc])
#     texts.extend(chunks)

# print(f"Total chunks created: {len(texts)}")

# # === Initialize embeddings ===
# print(f"Loading embedding model: {EMBEDDING_MODEL}")
# embeddings = HuggingFaceEmbeddings(
#     model_name=EMBEDDING_MODEL,
#     model_kwargs={"device": "cuda"}
# )

# # === Build FAISS vectorstore ===


# print(f"Total chunks created: {len(texts)}")

# # === Initialize embeddings (ĐÃ SỬA: Thêm normalize_embeddings) ===
# print(f"Loading embedding model: {EMBEDDING_MODEL}")

# # Cấu hình chuẩn hóa vector để sử dụng Cosine Similarity
# encode_kwargs = {'normalize_embeddings': True}

# embeddings = HuggingFaceEmbeddings(
#     model_name=EMBEDDING_MODEL,
#     model_kwargs={"device": "cuda"},
#     encode_kwargs=encode_kwargs # <-- BẮT BUỘC ĐỂ VECTOR ĐƯỢC CHUẨN HÓA
# )

# # === Build FAISS vectorstore (ĐÃ SỬA: Dùng Cosine và tối ưu hóa) ===
# vectorstore = None

# for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding and adding to FAISS (Cosine)"):
#     batch_docs = texts[i:i + BATCH_SIZE]

#     # 1. Tối ưu: Bỏ tính toán embedding thủ công (batch_embeddings)
#     # 2. Sử dụng add_documents thay vì add_texts để giữ metadata

#     if vectorstore is None:
#         # Lần đầu: Khởi tạo Vectorstore với Cosine Strategy
#         vectorstore = FAISS.from_documents(
#             documents=batch_docs, # Sử dụng documents
#             embedding=embeddings,
#             distance_strategy=DistanceStrategy.COSINE # <-- Faiss mặc định dùng L2 distance
#         )
#     else:
#         # Các lần sau: Thêm documents
#         # Hàm add_documents sẽ tự động gọi embeddings.embed_documents bên trong
#         vectorstore.add_documents(documents=batch_docs)

# # === Save FAISS index ===
# os.makedirs(OUTPUT_DIR, exist_ok=True)
# vectorstore.save_local(OUTPUT_DIR)
# print(f"FAISS vectorstore successfully saved to '{OUTPUT_DIR}'")

Loading text documents...
Total documents found: 16440
Documents selected for embedding: 16440


Splitting documents into chunks: 100%|██████████| 16440/16440 [02:07<00:00, 128.59it/s]


Total chunks created: 345297
Loading embedding model: keepitreal/vietnamese-sbert


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total chunks created: 345297
Loading embedding model: keepitreal/vietnamese-sbert


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Embedding and adding to FAISS (Cosine): 100%|██████████| 2698/2698 [1:31:32<00:00,  2.04s/it]


FAISS vectorstore successfully saved to './drive/MyDrive/LawChatbot/vietnamese-sbert_1024_cosine/'


---

## Phần 5: Retrieval — Tìm kiếm ngữ cảnh

**Thay đổi so với baseline:**

- Baseline chỉ dùng FAISS vector search (dense only, top-k=5)
- Improvement dùng Hybrid Search = FAISS (dense) + BM25 (lexical) với trọng số alpha
- Thêm tùy chọn CrossEncoder Reranking để sắp xếp lại top candidates
- Để tiết kiệm RAM/VRAM trên Colab, context đã được tính sẵn và lưu vào `res_with_context.csv`


* Vì colab không đủ RAM/VRAM để load vector store + embedding + llm và  rất dễ bị overload memory RAM/VRAM, đề xuất

* ==> để tiết kiệm bộ nhớ, tạo thêm các cột context trong file res.csv để lúc rag thay vì retrieve trong vector store, thì ta lấy context ta đã chuẩn bị sẵn

### 5.1 Tải file `res_with_context.csv` (đã retrieve sẵn)

File này chứa 100 câu hỏi đầu tiên từ RES dataset, với 3 cột context đã được tính trước:

- `context_baseline`: retrieval bằng FAISS thuần (giống baseline)
- `context_hybrid`: FAISS + BM25 hybrid search
- `context_hybrid_rerank`: FAISS + BM25 + CrossEncoder reranking


In [31]:
from google.colab import drive
import pandas as pd
import os

res_path = '/content/drive/MyDrive/LawChatbot/res_with_context.csv'
file_id = "1ljZzQzxfs5Or9nKcPmAIohTnC5-zpHMo" # file chứa 100 dòng đầu tiên trong file res.csv ban đầu
url = f"https://drive.google.com/uc?id={file_id}"

if not os.path.exists(res_path):
    df = pd.read_csv(url)
    df.to_csv(res_path, index=False)
    print("Đã tải và lưu file.")
else:
    df = pd.read_csv(res_path)
    print("File đã tồn tại, dùng file local.")


Đã tải và lưu file.


In [32]:
df = pd.read_csv(res_path)
df.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Câu Hỏi,Trả lời,Số hiệu VBPL (Trích xuất),Câu trả lời của URAx-Law,Reference có đúng không?,Trả lời có đúng không?,Unnamed: 6,...,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,context_hybrid,context_hybrid_rerank,ans_context_hybrid,ans_context_hybrid_rerank,context_baseline,ans_baseline
0,0,0,0,"Soft OTP là hình thức xác thực thế nào, gồm nh...",Căn cứ tại điểm d khoản 3 Điều 11 Thông tư 50/...,50/2024/TT-NHNN,### Soft OTP là gì?\n \n Soft OTP (One Time Pa...,1.0,1.0,NaN,...,NaN,NaN,Hạng mục,Số lượng,người sử dụng. - phân tích yêu cầu. - chuyển...,loại phần mềm và các thuật ngữ chuyên ngành c...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...,"và quản lý tổ chức bộ máy, biên chế, khai thác...",\nBạn là một trợ lý thông minh. Trả lời câu hỏ...
1,1,1,1,Những nội dung tối thiểu gì mà tổ chức cung cấ...,Căn cứ tại khoản 2 Điều 17 Thông tư 50/2024/TT...,50/2024/TT-NHNN,"Theo quy định, tổ chức cung cấp dịch vụ Online...",1.0,1.0,NaN,...,NaN,NaN,Tổng số câu hỏi,2748,"bến nghé, quận 1. 2. không thu phí gửi phương...",qua cổng dịch vụ công của tỉnh hoặc cổng dịch...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...,"hình ảnh với nhiều màu sắc phong phú, đa dạng ...",\nBạn là một trợ lý thông minh. Trả lời câu hỏ...
2,2,2,2,"Theo quy định hiện hành, đơn vị cung cấp dịch ...",Căn cứ theo Điều 15 Thông tư 50/2024/TT-NHNN q...,50/2024/TT-NHNN,### Hệ thống giám sát và theo dõi hoạt động tr...,1.0,1.0,NaN,...,NaN,NaN,Số câu trả lời được,132,và quy trình vận hành tiêu chuẩn · sử dụng cá...,và quy trình vận hành tiêu chuẩn · sử dụng cá...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...,"mối rà soát, kiểm soát, công khai thủ tục hành...",\nBạn là một trợ lý thông minh. Trả lời câu hỏ...
3,3,3,3,"Khái niệm ""hệ quản trị cơ sở dữ liệu"" được Thô...",Căn cứ tại khoản 10 Điều 2 Thông tư 50/2024/TT...,50/2024/TT-NHNN,"**Khái niệm ""hệ quản trị cơ sở dữ liệu""**\n \n...",1.0,1.0,NaN,...,NaN,NaN,Tổng điểm của số câu trả lời đúng reference,122,"công chức, viên chức ngân hàng nhà nước do thố...",tin của cơ sở dữ liệu quốc gia ... (id: 565266...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...,năm 2020 về quy chế quản lý phần mềm quản lý h...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...
4,4,4,4,"Từ ngày 01/01/2025, các yêu cầu bắt buộc đối v...",Căn cứ theo Điều 6 Thông tư 50/2024/TT-NHNN qu...,50/2024/TT-NHNN,"Từ ngày 01/01/2025, theo Thông tư số 50/2024/T...",1.0,1.0,NaN,...,NaN,NaN,Tổng điểm câu trả lời đúng,132,tìm người vào cơ sở dữ liệu được thu thập vào ...,của ngân hàng giám sát 1. ngân hàng giám sát ...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...,nghị chấp thuận danh sách dự kiến nhân sự của ...,\nBạn là một trợ lý thông minh. Trả lời câu hỏ...


* Nếu đã tải file res_with_context.csv sẵn rồi thì ta có thể tới phần generation luôn

### 5.2 (Tùy chọn) Tự build file `res_with_context.csv`

Chạy các cell bên dưới nếu muốn tự build context từ đầu (cần có vector store và đủ VRAM).


In [ ]:
# !pip install rank_bm25

#### 5.2.1 Khởi tạo `HybridFAISSBM25Searcher`

**Thay đổi so với baseline:** Class này tích hợp FAISS dense search và BM25 lexical search.

Điểm số cuối = `alpha x dense_score + (1 - alpha) x bm25_score`

Reranker sử dụng `BAAI/bge-reranker-large` (CrossEncoder) để sắp xếp lại kết quả.


In [ ]:
# # Build hybrid search(faiss+bm25) + rerank(optional)
# import os
# import re
# import pickle
# import numpy as np
# import torch

# from rank_bm25 import BM25Okapi
# from sentence_transformers import CrossEncoder

# from langchain_community.vectorstores import FAISS
# from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain_community.vectorstores.utils import DistanceStrategy

# def fast_tokenize(text: str):
#         text = text.lower()
#         text = re.sub(r"[^\w\s]", " ", text)
#         return text.split()

# class HybridFAISSBM25Searcher:
#     def __init__(
#         self,
#         faiss_path: str,
#         embedding_model_name: str,
#         rerank_model: str = None
#     ):
#         self.faiss_path = faiss_path
#         self.embedding_model_name = embedding_model_name
#         self.rerank_model_name = rerank_model

#         self._load_faiss()
#         self._build_bm25()
#         self._load_reranker()

#     def _load_faiss(self):
#         self.embeddings = HuggingFaceEmbeddings(
#             model_name=self.embedding_model_name,
#             model_kwargs={"device": "cuda"},
#             encode_kwargs={"normalize_embeddings": True}
#         )

#         self.faiss_db = FAISS.load_local(
#             self.faiss_path,
#             self.embeddings,
#             allow_dangerous_deserialization=True,
#             distance_strategy=DistanceStrategy.COSINE
#         )

#     def _build_bm25(self):
#         docs = list(self.faiss_db.docstore._dict.values())

#         self.texts = [doc.page_content for doc in docs]

#         # LUÔN gán doc_id
#         for i, doc in enumerate(docs):
#             doc.metadata["doc_id"] = i

#         tokenized_corpus = [
#             fast_tokenize(t) for t in self.texts
#         ]
#         self.bm25 = BM25Okapi(tokenized_corpus)

#     def _load_reranker(self):
#         self.reranker = None
#         if self.rerank_model_name:
#             self.reranker = CrossEncoder(
#                 self.rerank_model_name,
#                 max_length=512,
#                 device="cuda" if torch.cuda.is_available() else "cpu"
#             )
#     def _hybrid_search(
#         self,
#         query: str,
#         top_k: int,
#         dense_k: int,
#         alpha: float
#     ):
#         dense_results = self.faiss_db.similarity_search_with_score(
#             query,
#             k=dense_k
#         )

#         q_tokens = fast_tokenize(query)
#         bm25_scores = self.bm25.get_scores(q_tokens)

#         results = []

#         for doc, distance in dense_results:
#             dense_score = 1 - distance  # COSINE similarity
#             idx = doc.metadata["doc_id"]
#             lexical_score = bm25_scores[idx]

#             score = alpha * dense_score + (1 - alpha) * lexical_score
#             results.append((score, doc))

#         results.sort(key=lambda x: x[0], reverse=True)
#         return results[:top_k]
#     def _rerank(self, query: str, docs, top_k: int):
#         pairs = [(query, doc.page_content) for doc in docs]
#         scores = self.reranker.predict(pairs)

#         ranked = sorted(
#             zip(scores, docs),
#             key=lambda x: x[0],
#             reverse=True
#         )
#         return ranked[:top_k]
#     def search(
#         self,
#         query: str,
#         top_k: int = 5,
#         dense_k: int = 30,
#         alpha: float = 0.7,
#         use_rerank: bool = False,
#         rerank_k: int = 10
#     ):
#         """
#         use_rerank = True  → FAISS + BM25 + RERANK
#         use_rerank = False → FAISS + BM25
#         """

#         hybrid_results = self._hybrid_search(
#             query=query,
#             top_k=rerank_k if use_rerank else top_k,
#             dense_k=dense_k,
#             alpha=alpha
#         )

#         if not use_rerank or self.reranker is None:
#             return hybrid_results

#         docs_for_rerank = [doc for _, doc in hybrid_results]
#         return self._rerank(query, docs_for_rerank, top_k)


# searcher = HybridFAISSBM25Searcher(
#     faiss_path="/content/drive/MyDrive/LawChatbot/vietnamese-sbert_1024_cosine/",
#     embedding_model_name="keepitreal/vietnamese-sbert",
#     rerank_model="BAAI/bge-reranker-large"
# )



/tmp/ipykernel_38017/2026438370.py:36: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

#### 5.2.2 Tạo và lưu file `res_with_context.csv`


In [ ]:
# # Tạo file res_with_context.csv(tạo sẵn các cột context)
# # file này chỉ chứa 100 dòng đầu trong file res.csv
# import pandas as pd
# from tqdm import tqdm

# # ================= CONFIG =================
# INPUT_CSV =  '/content/drive/MyDrive/LawChatbot/RES.csv'
# OUTPUT_CSV =  '/content/drive/MyDrive/LawChatbot/res_with_context_v1.csv'

# TOP_K = 5
# DENSE_K = 30
# RERANK_K = 10
# ALPHA = 0.7
# # ==========================================


# def build_context(results):
#     """
#     results:
#       - hybrid: [(score, Document)]
#       - rerank: [(score, Document)]
#     """
#     context_texts = [doc.page_content for _, doc in results]
#     return "\n".join(context_texts)


# # ================= LOAD SEARCHER =================
# # searcher = HybridFAISSBM25Searcher(
# #     faiss_path="/content/drive/MyDrive/LawChatbot/vietnamese-sbert_1024_cosine/",
# #     embedding_model_name="keepitreal/vietnamese-sbert",
# #     rerank_model="BAAI/bge-reranker-large"
# # )

# # ================= LOAD CSV =================
# df = pd.read_csv(INPUT_CSV).head(100)
# contexts_hybrid = []
# contexts_rerank = []

# # ================= PROCESS =================
# for question in tqdm(df["Câu Hỏi"], desc="Building contexts"):
#     # Hybrid only
#     hybrid_results = searcher.search(
#         query=question,
#         top_k=TOP_K,
#         dense_k=DENSE_K,
#         alpha=ALPHA,
#         use_rerank=False
#     )

#     context_hybrid = build_context(hybrid_results)
#     contexts_hybrid.append(context_hybrid)

#     # Hybrid + Rerank
#     hybrid_rerank_results = searcher.search(
#         query=question,
#         top_k=TOP_K,
#         dense_k=DENSE_K,
#         alpha=ALPHA,
#         use_rerank=True,
#         rerank_k=RERANK_K
#     )

#     context_rerank = build_context(hybrid_rerank_results)
#     contexts_rerank.append(context_rerank)

# # ================= SAVE =================
# df["context_hybrid"] = contexts_hybrid
# df["context_hybrid_rerank"] = contexts_rerank

# df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# print(f"Done! Saved to {OUTPUT_CSV}")

Building contexts: 100%|██████████| 100/100 [21:00<00:00, 12.61s/it]

Done! Saved to /content/drive/MyDrive/LawChatbot/res_with_context_v1.csv


---

## Phần 6: Generation — Sinh câu trả lời

**Thay đổi so với baseline:**

- LLM nâng cấp từ `Qwen/Qwen1.5-1.8B` lên `Qwen/Qwen3-1.7B` — mô hình mới hơn, hiệu năng tốt hơn
- Prompt được cấu trúc rõ ràng hơn với hướng dẫn không được bịa đặt thông tin
- Hỗ trợ sinh câu trả lời cho 3 loại context: baseline, hybrid, hybrid+rerank


### Sử dụng kết quả đã generate từ res_with_context.csv

* Tải file res_with_context.csv như hướng dẫn ở trên

In [34]:
!pip install langchain_huggingface

### 6.1 Đánh giá nhanh bằng Cosine Similarity

So sánh embedding của câu trả lời được sinh ra với ground truth (`Trả lời`) trong dataset.


In [35]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# =============================
# 1. LOAD DATA
# =============================

df = pd.read_csv("/content/drive/MyDrive/LawChatbot/RES.csv").head(30)
col1 = "Trả lời"
col2 = "Câu trả lời của URAx-Law"

# =============================
# 2. DANH SÁCH MODEL CẦN TEST
# =============================

embedding_models = {
    "huyydangg/DEk21_hcmute_embedding": "huyydangg/DEk21_hcmute_embedding",
    "AITeamVN/Vietnamese_Embedding": "AITeamVN/Vietnamese_Embedding",
    "BAAI/bge-m3": "BAAI/bge-m3",
    "keepitreal/vietnamese-sbert": "keepitreal/vietnamese-sbert"
}

# =============================
# 3. TÍNH COSINE SIMILARITY
# =============================

results = pd.DataFrame()
results["Trả lời"] = df[col1]
results["Câu trả lời của URAx-Law"] = df[col2]

for model_name, model_path in embedding_models.items():
    print(f"\n Loading model: {model_name} ...")
    model = SentenceTransformer(model_path)

    sims = []
    for a, b in tqdm(zip(df[col1], df[col2]), total=len(df)):
        emb_a = model.encode(str(a), convert_to_numpy=True)
        emb_b = model.encode(str(b), convert_to_numpy=True)
        cos_sim = cosine_similarity([emb_a], [emb_b])[0][0]
        sims.append(cos_sim)

    results[f"cosine_{model_name}"] = sims
    # print(results)
# =============================
# 4. TÍNH GIÁ TRỊ TRUNG BÌNH CHO TỪNG MÔ HÌNH
# =============================

avg_scores = {}
for model_name in embedding_models.keys():
    col = f"cosine_{model_name}"
    avg_scores[model_name] = results[col].mean()

avg_df = pd.DataFrame.from_dict(avg_scores, orient="index", columns=["mean_cosine_similarity"])
avg_df = avg_df.sort_values(by="mean_cosine_similarity", ascending=False)

print("\n=====================")
print(" Average Cosine Similarity per Model")
print("=====================\n")
print(avg_df)
# =============================
# 5. LƯU FILE KẾT QUẢ
# =============================

results.to_csv("compare_embeddings_results.csv", index=False)
avg_df.to_csv("average_cosine_similarity.csv")

print("\n DONE! Saved:")
print(" compare_embeddings_results.csv")
print(" average_cosine_similarity.csv")



🔄 Loading model: huyydangg/DEk21_hcmute_embedding ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

100%|██████████| 30/30 [00:43<00:00,  1.47s/it]



🔄 Loading model: AITeamVN/Vietnamese_Embedding ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

100%|██████████| 30/30 [03:24<00:00,  6.81s/it]



🔄 Loading model: keepitreal/vietnamese-sbert ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 30/30 [00:43<00:00,  1.45s/it]


 Average Cosine Similarity per Model

                                  mean_cosine_similarity
BAAI/bge-m3                                     0.891451
keepitreal/vietnamese-sbert                     0.848082
huyydangg/DEk21_hcmute_embedding                0.814539
AITeamVN/Vietnamese_Embedding                   0.773261

 DONE! Saved:
 compare_embeddings_results.csv
 average_cosine_similarity.csv


### 6.2 Evaluation — Đánh giá toàn diện các metrics

Tính toán đầy đủ 5 metrics cho cả 3 phương pháp retrieval:

- **Cosine Similarity**: Độ tương đồng ngữ nghĩa
- **Jaccard Similarity**: Mức độ trùng lặp từ vựng
- **Token Overlap**: Tỷ lệ từ trong ground truth xuất hiện trong câu trả lời
- **BLEU Score**: N-gram precision
- **ROUGE-L**: Longest Common Subsequence


In [36]:
import os
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import Counter
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings # Changed import path
from langchain_huggingface import HuggingFacePipeline
from langchain_google_genai import ChatGoogleGenerativeAI
from sklearn.metrics.pairwise import cosine_similarity


def normalize_text(text: str) -> str:
    text = str(text).lower().replace("\n", " ").strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def cosine_similarity_score(pred: str, true: str, emb_model) -> float:
    if not pred or not true:
        return 0.0
    pred_vec = emb_model.embed_query(pred)
    true_vec = emb_model.embed_query(true)
    return float(cosine_similarity([pred_vec], [true_vec])[0][0])

def jaccard_similarity(pred: str, true: str) -> float:
    pred_words = set(normalize_text(pred).split())
    true_words = set(normalize_text(true).split())
    return len(pred_words & true_words) / len(pred_words | true_words) if pred_words and true_words else 0.0

def token_overlap_score(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not true_words:
        return 0.0
    return sum((Counter(pred_words) & Counter(true_words)).values()) / len(true_words)

def bleu_score_simple(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not pred_words or not true_words:
        return 0.0
    overlap = sum((Counter(pred_words) & Counter(true_words)).values())
    precision = overlap / len(pred_words)
    bp = 1.0 if len(pred_words) >= len(true_words) else np.exp(1 - len(true_words)/len(pred_words))
    return bp * precision

def rouge_l_score(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not pred_words or not true_words:
        return 0.0
    m, n = len(pred_words), len(true_words)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if pred_words[i-1] == true_words[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    precision = lcs / m
    recall = lcs / n
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

def semantic_containment(pred: str, true: str) -> float:
    pred_norm = normalize_text(pred)
    true_norm = normalize_text(true)
    if not true_norm:
        return 0.0
    if true_norm in pred_norm:
        return 1.0
    pred_words = set(pred_norm.split())
    true_words = set(true_norm.split())
    return 1.0 if true_words.issubset(pred_words) else 0.

embedding_model_name = "keepitreal/vietnamese-sbert"
encode_kwargs = {'normalize_embeddings': True}
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    encode_kwargs=encode_kwargs  # Changed from model_kwargs to encode_kwargs
)

df = pd.read_csv("/content/drive/MyDrive/LawChatbot/res_with_context.csv")
print(f"Total number of questions: {len(df)}")


# ==================== EVALUATION ====================
for i in ['ans_baseline', 'ans_context_hybrid', 'ans_context_hybrid_rerank']:
  results = []
  print(f"----------{i}-----------")
  for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating RAG"):
      ground_truth = str(row["Trả lời"]).strip()

      try:
          response = df.at[idx, i]

      except Exception as e:
          print(f"Error at question {idx}: {e}")
          response = ""

      metrics = {
          "Cosine_Similarity": cosine_similarity_score(response, ground_truth, embeddings),
          "Jaccard_Similarity": jaccard_similarity(response, ground_truth),
          "Token_Overlap": token_overlap_score(response, ground_truth),
          "BLEU_Score": bleu_score_simple(response, ground_truth),
          "ROUGE_L": rouge_l_score(response, ground_truth)
      }

      results.append(metrics)

  results_df = pd.DataFrame(results)
  mean_metrics = results_df.mean().to_dict()

  print("\n=== FINAL RAG EVALUATION SUMMARY ===")
  for k, v in mean_metrics.items():
      print(f"{k}: {v:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total number of questions: 100
----------ans_baseline-----------


Evaluating RAG: 100%|██████████| 100/100 [02:34<00:00,  1.55s/it]



=== FINAL RAG EVALUATION SUMMARY ===
Cosine_Similarity: 0.6867
Jaccard_Similarity: 0.1957
Token_Overlap: 0.7464
BLEU_Score: 0.1047
ROUGE_L: 0.0902
----------ans_context_hybrid-----------


Evaluating RAG: 100%|██████████| 100/100 [02:21<00:00,  1.42s/it]



=== FINAL RAG EVALUATION SUMMARY ===
Cosine_Similarity: 0.6993
Jaccard_Similarity: 0.2413
Token_Overlap: 0.7149
BLEU_Score: 0.1685
ROUGE_L: 0.1333
----------ans_context_hybrid_rerank-----------


Evaluating RAG: 100%|██████████| 100/100 [02:22<00:00,  1.43s/it]


=== FINAL RAG EVALUATION SUMMARY ===
Cosine_Similarity: 0.7026
Jaccard_Similarity: 0.2447
Token_Overlap: 0.7173
BLEU_Score: 0.1691
ROUGE_L: 0.1310


---

## Phần 7: Demo — Minh họa kết quả


### 7.1 Hiển thị so sánh câu trả lời của 3 phương pháp trên 5 câu hỏi mẫu.


In [40]:
# ===================== RAG DEMONSTRATION (PLAIN TEXT) =====================

import pandas as pd

# ===================== LOAD DATA =====================
CSV_PATH = "/content/drive/MyDrive/LawChatbot/res_with_context.csv"
df = pd.read_csv(CSV_PATH)

print(f"Total number of questions in dataset: {len(df)}")
# ===================== SELECT 5 QUESTIONS =====================
sample_indices = [0, 3, 7, 12, 18]
sample_df = df.loc[sample_indices]

# ===================== HELPER =====================
def print_from_question(text: str) -> str:
    if not isinstance(text, str):
        return ""
    idx = text.find("Câu hỏi:")
    return text[idx:].strip() if idx != -1 else text.strip()

# ===================== DEMONSTRATION =====================
for i, (_, row) in enumerate(sample_df.iterrows(), start=1):
    print("=" * 80)
    print(f"Câu hỏi {i}:")
    print(row["Câu Hỏi"])
    print()

    print("Trả lời đúng:")
    print(row["Trả lời"])
    print()
    print("-" * 80)
    print("Baseline:")
    print(print_from_question(row["ans_baseline"]))
    print()
    print("-" * 80)
    print("Hybrid Search:")
    print(print_from_question(row["ans_context_hybrid"]))
    print()
    print("-" * 80)
    print("Hybrid Search + Rerank:")
    print(print_from_question(row["ans_context_hybrid_rerank"]))
    print()




Total number of questions in dataset: 100
Câu hỏi 1:
Soft OTP là hình thức xác thực thế nào, gồm những loại nào và phần mềm Soft OTP cần tuân thủ những yêu cầu cụ thể gì theo quy định?

Trả lời đúng:
Căn cứ tại điểm d khoản 3 Điều 11 Thông tư 50/2024/TT-NHNN quy định về Soft OTP là hình thức xác nhận thông qua mã OTP được tạo bởi phần mềm cài đặt trên thiết bị di động của khách hàng, phần mềm Soft OTP có thể là phần mềm độc lập hoặc được tích hợp với phần mềm ứng dụng Mobile Banking.
- Soft OTP có 02 loại:
+ Soft OTP loại cơ bản: Mã OTP được sinh ngẫu nhiên theo thời gian, đồng bộ với hệ thống Online Banking;
+ Soft OTP loại nâng cao: Mã OTP được tạo kết hợp với mã của từng giao dịch, khi thực hiện giao dịch, hệ thống Online Banking tạo ra một mã giao dịch thông báo cho khách hàng hoặc truyền cho phần mềm Soft OTP, khách hàng hoặc phần mềm Soft OTP tự động nhập mã giao dịch vào phần mềm Soft OTP để phần mềm Soft OTP tạo ra mã OTP.
- Soft OTP phải đáp ứng yêu cầu như sau:
+ Trường hợp p

### 7.2 Prompt câu hỏi để xem câu trả lời: Nhập index của câu hỏi theo file RES


In [45]:
# ===================== DEMO: Chọn câu hỏi theo số thứ tự =====================

from ipywidgets import widgets
from IPython.display import display, clear_output

index_input = widgets.BoundedIntText(
    value=0,
    min=0,
    max=len(df) - 1,
    step=1,
    description="Index:",
    layout=widgets.Layout(width="200px")
)

run_button = widgets.Button(
    description="Xem kết quả",
    button_style="primary",
    layout=widgets.Layout(width="150px")
)

output_area = widgets.Output()

def on_click(b):
    with output_area:
        clear_output()
        idx = index_input.value
        row = df.loc[idx]

        print("=" * 80)
        print(f"Index: {idx}")
        print()
        print("Cau hoi:")
        print(row["Câu Hỏi"])
        print()
        print("Tra loi dung:")
        print(row["Trả lời"])
        print()
        print("-" * 80)
        print("Baseline:")
        print(print_from_question(row["ans_baseline"]))
        print()
        print("-" * 80)
        print("Hybrid Search:")
        print(print_from_question(row["ans_context_hybrid"]))
        print()
        print("-" * 80)
        print("Hybrid Search + Rerank:")
        print(print_from_question(row["ans_context_hybrid_rerank"]))
        print("=" * 80)

run_button.on_click(on_click)

display(widgets.VBox([index_input, run_button, output_area]))